# Phase 1: Data Cleaning & Quality Assessment
## E-Commerce Revenue & Customer Lifecycle Analytics

**Analyst:** Utsav Khadka  
**Dataset:** UCI Online Retail II (Year 2010–2011 sheet)  
**Goal:** Clean raw transaction data so it's ready for SQL analysis, EDA, and dashboarding

---

### Business Context
The VP of Sales asked: *"Revenue looks flat. Are we losing customers?"*  
Before we can answer that, we need clean, trustworthy data.  
Garbage in = garbage out. Every analysis we do in Phase 2 and 3 depends on this step.

---

### What This Notebook Does (Step by Step)
1. Load the raw dataset and profile it
2. Identify and separate cancellations
3. Handle missing Customer IDs
4. Remove bad prices (£0 and negative)
5. Remove duplicate rows
6. Create derived columns (Revenue, date parts)
7. Save the cleaned dataset
8. Write a data quality report

---
## Step 0: Import Libraries

**WHY:** We need to tell Python which tools we're using before we can use them.  
Think of this like gathering your tools before starting a job.

- `pandas` → our main tool for working with data tables
- `numpy` → for math operations
- `warnings` → to silence non-critical warnings that clutter output

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Show all columns when printing a DataFrame — never truncate
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully')
print(f'Pandas version: {pd.__version__}')

Libraries loaded successfully
Pandas version: 3.0.3


---
## Step 1: Load the Raw Dataset

**WHY:** We're loading only the `Year 2010-2011` sheet.  

**Decision:** Why not both sheets?  
- The `Year 2010-2011` sheet matches the original UCI Online Retail dataset exactly
- It has ~541,000 rows — large enough for all our analyses
- Combining both sheets would require extra cross-year cleaning with diminishing returns

**Note:** This cell takes 20-30 seconds — the file is 44MB. That's normal.

In [2]:
# Load only the Year 2010-2011 sheet from the Excel file
print('Loading dataset... (this takes ~30 seconds for a 44MB Excel file)')

df_raw = pd.read_excel(
    '../data/raw/online_retail_II.xlsx',
    sheet_name='Year 2010-2011'   # Only loading this sheet
)

print(f'Dataset loaded successfully!')
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

Loading dataset... (this takes ~30 seconds for a 44MB Excel file)
Dataset loaded successfully!
Shape: 541,910 rows x 8 columns


---
## Step 2: First Look at the Data

**WHY:** Before touching anything, we look at the raw data exactly as it is.  
This is called **profiling** — understanding what you have before deciding what to fix.

A real analyst never jumps straight to cleaning. You look first.

In [3]:
# Look at the first 5 rows — understand the structure
print('=== FIRST 5 ROWS ===')
df_raw.head()

=== FIRST 5 ROWS ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom


In [4]:
# Check column names and data types
# WHY: We need to know if dates loaded as strings or actual dates,
#      if numbers loaded as text, etc.
print('=== COLUMN NAMES & DATA TYPES ===')
print(df_raw.dtypes)
print(f'\nTotal columns: {len(df_raw.columns)}')

=== COLUMN NAMES & DATA TYPES ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Total columns: 8


In [5]:
# Check for missing values in each column
# WHY: Missing data affects every analysis downstream
#      We need to know HOW MUCH is missing before deciding what to do
print('=== MISSING VALUES PER COLUMN ===')
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_report)
print(f'\nTotal rows in raw dataset: {len(df_raw):,}')

=== MISSING VALUES PER COLUMN ===
             Missing Count  Missing %
Invoice                  0       0.00
StockCode                0       0.00
Description           1454       0.27
Quantity                 0       0.00
InvoiceDate              0       0.00
Price                    0       0.00
Customer ID         135080      24.93
Country                  0       0.00

Total rows in raw dataset: 541,910


In [6]:
# Check basic statistics for numeric columns
# WHY: This reveals outliers, negative values, and impossible numbers
#      e.g. Quantity = -80 or Price = 0 immediately stands out here
print('=== BASIC STATISTICS ===')
df_raw.describe()

=== BASIC STATISTICS ===


,Quantity,InvoiceDate,Price,Customer ID
count,541910.00,541910,541910.00,406830.00
mean,9.55,2011-07-04 13:35:22.342307,4.61,15287.68
min,-80995.00,2010-12-01 08:26:00,-11062.06,12346.00
25%,1.00,2011-03-28 11:34:00,1.25,13953.00
50%,3.00,2011-07-19 17:17:00,2.08,15152.00
75%,10.00,2011-10-19 11:27:00,4.13,16791.00
max,80995.00,2011-12-09 12:50:00,38970.00,18287.00
std,218.08,NaN,96.76,1713.60


In [7]:
# Check date range of the dataset
# WHY: Confirms we loaded the right sheet (should be Dec 2010 - Dec 2011)
print('=== DATE RANGE ===')
print(f'Earliest date: {df_raw["InvoiceDate"].min()}')
print(f'Latest date:   {df_raw["InvoiceDate"].max()}')
print(f'\nUnique countries: {df_raw["Country"].nunique()}')
print(f'Unique customers: {df_raw["Customer ID"].nunique():,}')
print(f'Unique products:  {df_raw["StockCode"].nunique():,}')

=== DATE RANGE ===
Earliest date: 2010-12-01 08:26:00
Latest date:   2011-12-09 12:50:00

Unique countries: 38
Unique customers: 4,372
Unique products:  4,070


---
## Step 3: Standardize Column Names

**WHY:** The column names in this dataset have spaces and inconsistent formatting:  
- `Customer ID` has a space → makes it annoying to reference in code  
- `Invoice` should be `InvoiceNo` to match industry convention  
- `Price` should be `UnitPrice` to be self-explanatory  

**Rule:** Consistent column names = fewer bugs in SQL and Python.

In [8]:
# Work on a copy — NEVER modify the raw dataframe
# WHY: If we make a mistake, we can always go back to df_raw
#      without re-loading the 44MB file
df = df_raw.copy()

# Rename columns to clean, consistent names
df = df.rename(columns={
    'Invoice'    : 'InvoiceNo',
    'Price'      : 'UnitPrice',
    'Customer ID': 'CustomerID'
})

print('=== COLUMN NAMES AFTER RENAMING ===')
print(df.columns.tolist())
print(f'\nShape unchanged: {df.shape}')

=== COLUMN NAMES AFTER RENAMING ===
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Shape unchanged: (541910, 8)


---
## Step 4: Identify & Flag Cancellations

**WHY:** Cancellations are hidden inside the dataset.  
They look like normal rows EXCEPT the InvoiceNo starts with the letter `C`.

**Critical Decision — Why we FLAG instead of DELETE:**  
Cancellation rate is itself a business metric.  
If 15% of orders are being cancelled, the VP of Sales needs to know that.  
Silently deleting them would hide a real business problem.

**What we do:**
- Add a column `is_cancelled` = True/False
- Keep all rows in the main dataset
- Use the flag to include/exclude cancellations depending on the analysis

In [9]:
# Flag cancellations — InvoiceNo starting with 'C' = cancellation
df['is_cancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')

# Count cancellations
cancellation_counts = df['is_cancelled'].value_counts()
cancellation_rate = (df['is_cancelled'].sum() / len(df) * 100).round(2)

print('=== CANCELLATION ANALYSIS ===')
print(f'Normal transactions: {cancellation_counts[False]:,}')
print(f'Cancellations:       {cancellation_counts[True]:,}')
print(f'Cancellation rate:   {cancellation_rate}%')
print(f'\nBusiness Insight: {cancellation_rate}% of all orders were cancelled or returned')

=== CANCELLATION ANALYSIS ===
Normal transactions: 532,622
Cancellations:       9,288
Cancellation rate:   1.71%

Business Insight: 1.71% of all orders were cancelled or returned


---
## Step 5: Handle Missing Customer IDs

**WHY:** About 25% of rows have no CustomerID — these are likely guest checkouts  
(customers who bought without creating an account).

**Decision — Two approaches for two different analyses:**

| Analysis Type | CustomerID Nulls | Why |
|--------------|-----------------|-----|
| Revenue / Product / Geographic | KEEP | Revenue happened regardless of who bought |
| Cohort Analysis / RFM | DROP | Can't track a customer with no ID |

**What we do:**  
- Quantify the revenue impact of dropping nulls  
- Keep a note of the decision for the quality report  
- Create `df_customers` (no nulls) for customer-level analysis

In [10]:
# First — quantify the revenue impact of null CustomerIDs
# WHY: We need to know what % of revenue has no customer info

# Calculate revenue for all rows first (even messy ones)
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Split into null vs non-null CustomerID
null_customers    = df[df['CustomerID'].isnull()]
non_null_customers = df[df['CustomerID'].notnull()]

total_revenue    = df[~df['is_cancelled']]['Revenue'].sum()
null_revenue     = null_customers[~null_customers['is_cancelled']]['Revenue'].sum()
null_revenue_pct = (null_revenue / total_revenue * 100).round(2)

print('=== MISSING CUSTOMER ID IMPACT ===')
print(f'Total rows:                {len(df):,}')
print(f'Rows with CustomerID:      {len(non_null_customers):,} ({len(non_null_customers)/len(df)*100:.1f}%)')
print(f'Rows WITHOUT CustomerID:   {len(null_customers):,} ({len(null_customers)/len(df)*100:.1f}%)')
print(f'\nRevenue from null IDs:     £{null_revenue:,.2f} ({null_revenue_pct}% of total)')
print(f'\nDecision: Drop nulls for customer analysis. Keep for revenue/product analysis.')

=== MISSING CUSTOMER ID IMPACT ===
Total rows:                541,910
Rows with CustomerID:      406,830 (75.1%)
Rows WITHOUT CustomerID:   135,080 (24.9%)

Revenue from null IDs:     £1,733,152.52 (16.28% of total)

Decision: Drop nulls for customer analysis. Keep for revenue/product analysis.


---
## Step 6: Remove Bad Prices

**WHY:** Some rows have UnitPrice = 0 or negative.  
These are NOT real customer transactions — they are typically:  
- Internal test orders placed by the company  
- Manual adjustments or corrections  
- Data entry errors  

**Decision:** Remove rows where UnitPrice <= 0 from the clean dataset.  
We first quantify how many rows this affects.

In [11]:
# Identify rows with bad prices
bad_price_rows = df[df['UnitPrice'] <= 0]

print('=== BAD PRICE ANALYSIS ===')
print(f'Rows with UnitPrice <= 0: {len(bad_price_rows):,}')
print(f'That is {len(bad_price_rows)/len(df)*100:.2f}% of all rows')
print(f'\nSample of bad price rows:')
bad_price_rows[['InvoiceNo','Description','Quantity','UnitPrice','CustomerID']].head(10)

=== BAD PRICE ANALYSIS ===
Rows with UnitPrice <= 0: 2,517
That is 0.46% of all rows

Sample of bad price rows:


,InvoiceNo,Description,Quantity,UnitPrice,CustomerID
622,536414,NaN,56,0.00,NaN
1510,536545,NaN,1,0.00,NaN
1985,536547,NaN,1,0.00,NaN
1986,536546,NaN,1,0.00,NaN
2022,536552,NaN,1,0.00,NaN
2023,536549,NaN,1,0.00,NaN
2024,536550,NaN,1,0.00,NaN
2025,536553,NaN,3,0.00,NaN
2026,536554,NaN,23,0.00,NaN
2406,536589,NaN,-10,0.00,NaN


---
## Step 7: Remove Duplicate Rows

**WHY:** Duplicate rows inflate every metric — revenue looks higher,  
customer counts look higher, everything is wrong.

**What counts as a duplicate?**  
An exact copy of a row — same InvoiceNo, StockCode, Quantity, Price, CustomerID, Date.

In [12]:
# Count duplicates before removing
duplicate_count = df.duplicated().sum()

print('=== DUPLICATE ROWS ===')
print(f'Duplicate rows found: {duplicate_count:,}')
print(f'That is {duplicate_count/len(df)*100:.2f}% of all rows')

=== DUPLICATE ROWS ===
Duplicate rows found: 5,268
That is 0.97% of all rows


---
## Step 8: Build the Clean Dataset

**WHY:** Now we apply ALL cleaning decisions together in one place.  
This is cleaner than modifying the dataframe step by step — we define ALL rules,  
then apply them once.

**Cleaning rules being applied:**
1. Remove duplicate rows
2. Remove rows where UnitPrice <= 0
3. Standardize Description text (strip whitespace, title case)
4. Convert CustomerID to integer (remove decimal .0)
5. Extract date parts from InvoiceDate

In [13]:
# Record row count before cleaning
rows_before = len(df)
print(f'Rows BEFORE cleaning: {rows_before:,}')
print('Applying cleaning rules...')

# Rule 1 — Remove exact duplicate rows
df_clean = df.drop_duplicates()
print(f'After removing duplicates:        {len(df_clean):,} rows (removed {rows_before - len(df_clean):,})')

# Rule 2 — Remove rows where UnitPrice <= 0
df_clean = df_clean[df_clean['UnitPrice'] > 0]
print(f'After removing bad prices:        {len(df_clean):,} rows (removed {rows_before - len(df_clean):,} total so far)')

# Rule 3 — Clean Description text
# WHY: Same product appears as 'WHITE HANGING HEART', 'white hanging heart', '  White Hanging Heart  '
#      Standardizing prevents triple-counting the same product
df_clean['Description'] = df_clean['Description'].astype(str).str.strip().str.title()

# Rule 4 — Convert CustomerID to integer
# WHY: It loaded as 12345.0 (float) — looks ugly, wastes memory
#      We keep nulls as-is (can't convert null to int without workaround)
df_clean['CustomerID'] = df_clean['CustomerID'].astype('Int64')  # Int64 supports nulls

# Rule 5 — Extract date parts from InvoiceDate
# WHY: SQL and Tableau need individual date components for time-series analysis
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Date']        = df_clean['InvoiceDate'].dt.date
df_clean['Year']        = df_clean['InvoiceDate'].dt.year
df_clean['Month']       = df_clean['InvoiceDate'].dt.month
df_clean['Month_Name']  = df_clean['InvoiceDate'].dt.strftime('%B')
df_clean['Day_of_Week'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour']        = df_clean['InvoiceDate'].dt.hour
df_clean['YearMonth']   = df_clean['InvoiceDate'].dt.to_period('M').astype(str)

# Rule 6 — Recalculate Revenue on clean data
# WHY: Revenue was calculated on raw data earlier — recalculate on clean data
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

rows_after = len(df_clean)
print(f'\nRows AFTER all cleaning:          {rows_after:,}')
print(f'Total rows removed:               {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.1f}%)')

Rows BEFORE cleaning: 541,910
Applying cleaning rules...
After removing duplicates:        536,642 rows (removed 5,268)
After removing bad prices:        534,130 rows (removed 7,780 total so far)

Rows AFTER all cleaning:          534,130
Total rows removed:               7,780 (1.4%)


---
## Step 9: Final Verification

**WHY:** Always verify your cleaning worked correctly.  
A real analyst never assumes — they check.

In [14]:
# Verify — no more bad prices
print('=== VERIFICATION CHECKS ===')
print(f'Rows with UnitPrice <= 0:    {(df_clean["UnitPrice"] <= 0).sum()} (should be 0)')
print(f'Duplicate rows:              {df_clean.duplicated().sum()} (should be 0)')
print(f'\nDate range after cleaning:')
print(f'  Earliest: {df_clean["InvoiceDate"].min()}')
print(f'  Latest:   {df_clean["InvoiceDate"].max()}')
print(f'\nFinal column list:')
print(df_clean.columns.tolist())
print(f'\nFirst 3 rows of clean data:')
df_clean.head(3)

=== VERIFICATION CHECKS ===
Rows with UnitPrice <= 0:    0 (should be 0)
Duplicate rows:              0 (should be 0)

Date range after cleaning:
  Earliest: 2010-12-01 08:26:00
  Latest:   2011-12-09 12:50:00

Final column list:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'is_cancelled', 'Revenue', 'Date', 'Year', 'Month', 'Month_Name', 'Day_of_Week', 'Hour', 'YearMonth']

First 3 rows of clean data:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,Revenue,Date,Year,Month,Month_Name,Day_of_Week,Hour,YearMonth
0,536365,85123A,White Hanging Heart T-Light Holder,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,15.30,2010-12-01,2010,12,December,Wednesday,8,2010-12
1,536365,71053,White Metal Lantern,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34,2010-12-01,2010,12,December,Wednesday,8,2010-12
2,536365,84406B,Cream Cupid Hearts Coat Hanger,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,22.00,2010-12-01,2010,12,December,Wednesday,8,2010-12


In [15]:
# Key business metrics from clean data (non-cancelled only)
df_sales = df_clean[~df_clean['is_cancelled']]

print('=== KEY BUSINESS METRICS (Clean Data, Non-Cancelled) ===')
print(f'Total Revenue:          £{df_sales["Revenue"].sum():,.2f}')
print(f'Total Orders:           {df_sales["InvoiceNo"].nunique():,}')
print(f'Total Customers:        {df_sales["CustomerID"].nunique():,}')
print(f'Total Products:         {df_sales["StockCode"].nunique():,}')
print(f'Total Countries:        {df_sales["Country"].nunique():,}')
print(f'Avg Order Value (AOV):  £{df_sales.groupby("InvoiceNo")["Revenue"].sum().mean():,.2f}')

=== KEY BUSINESS METRICS (Clean Data, Non-Cancelled) ===
Total Revenue:          £10,642,128.80
Total Orders:           19,960
Total Customers:        4,338
Total Products:         3,922
Total Countries:        38
Avg Order Value (AOV):  £533.17


---
## Step 10: Save the Cleaned Dataset

**WHY:** We save the clean data as a CSV file so:
- Phase 2 SQL analysis can read it directly with DuckDB
- Phase 3 EDA notebook loads clean data (not raw)
- We never have to re-run this cleaning notebook again

**Rule:** Raw data stays in `data/raw/` — clean data goes to `data/cleaned/`

In [16]:
# Save full cleaned dataset (includes cancellations — flagged)
output_path = '../data/cleaned/online_retail_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f'Clean dataset saved to: {output_path}')
print(f'File contains {len(df_clean):,} rows and {len(df_clean.columns)} columns')

Clean dataset saved to: ../data/cleaned/online_retail_cleaned.csv
File contains 534,130 rows and 17 columns


---
## Step 11: Data Quality Report

**WHY:** This is the deliverable for Phase 1.  
A real analyst always documents what they found and what they did about it.  
This report becomes part of the README and is referenced in the executive memo.

In [17]:
print('=' * 60)
print('       DATA QUALITY REPORT — PHASE 1')
print('       E-Commerce Revenue Analytics')
print('=' * 60)

print(f'''
DATASET
-------
Source:          UCI Online Retail II (Year 2010-2011 sheet)
Raw rows:        {rows_before:,}
Clean rows:      {rows_after:,}
Rows removed:    {rows_before - rows_after:,} ({(rows_before-rows_after)/rows_before*100:.1f}%)

ISSUES FOUND & DECISIONS MADE
------------------------------
1. CANCELLATIONS
   Found:    {df['is_cancelled'].sum():,} cancelled invoices ({cancellation_rate}% of all orders)
   Decision: Flagged with is_cancelled column. NOT deleted.
   Reason:   Cancellation rate is a business metric — hiding it is bad analysis.

2. MISSING CUSTOMER IDs
   Found:    {len(null_customers):,} rows without CustomerID ({len(null_customers)/len(df)*100:.1f}% of data)
   Decision: Keep for revenue/product analysis. Drop for cohort/RFM analysis.
   Reason:   Guest checkouts contribute real revenue but can't be tracked as customers.

3. BAD PRICES (UnitPrice <= 0)
   Found:    {len(bad_price_rows):,} rows
   Decision: Removed from clean dataset.
   Reason:   These are internal/test transactions with no business value.

4. DUPLICATE ROWS
   Found:    {duplicate_count:,} exact duplicate rows
   Decision: Removed.
   Reason:   Duplicates inflate every metric — revenue, orders, customers.

5. COLUMN STANDARDIZATION
   Renamed: Invoice → InvoiceNo, Price → UnitPrice, Customer ID → CustomerID
   Reason:  Consistent naming prevents errors across SQL and Python.

NEW COLUMNS CREATED
-------------------
Revenue      = Quantity * UnitPrice
is_cancelled = True if InvoiceNo starts with C
Date         = Date portion of InvoiceDate
Year         = Year extracted from InvoiceDate
Month        = Month number (1-12)
Month_Name   = Month name (January, February...)
Day_of_Week  = Day name (Monday, Tuesday...)
Hour         = Hour of transaction (0-23)
YearMonth    = Year-Month string (2010-12, 2011-01...)

OUTPUT
------
Clean file: data/cleaned/online_retail_cleaned.csv
''')
print('=' * 60)

       DATA QUALITY REPORT — PHASE 1
       E-Commerce Revenue Analytics

DATASET
-------
Source:          UCI Online Retail II (Year 2010-2011 sheet)
Raw rows:        541,910
Clean rows:      534,130
Rows removed:    7,780 (1.4%)

ISSUES FOUND & DECISIONS MADE
------------------------------
1. CANCELLATIONS
   Found:    9,288 cancelled invoices (1.71% of all orders)
   Decision: Flagged with is_cancelled column. NOT deleted.
   Reason:   Cancellation rate is a business metric — hiding it is bad analysis.

2. MISSING CUSTOMER IDs
   Found:    135,080 rows without CustomerID (24.9% of data)
   Decision: Keep for revenue/product analysis. Drop for cohort/RFM analysis.
   Reason:   Guest checkouts contribute real revenue but can't be tracked as customers.

3. BAD PRICES (UnitPrice <= 0)
   Found:    2,517 rows
   Decision: Removed from clean dataset.
   Reason:   These are internal/test transactions with no business value.

4. DUPLICATE ROWS
   Found:    5,268 exact duplicate rows
   Deci

---
## Phase 1 Complete ✅

**What we built:**
- A clean, analysis-ready dataset saved to `data/cleaned/`
- A documented record of every cleaning decision and why
- Key business metrics established as baseline

**What's next — Phase 2:**
- Load `online_retail_cleaned.csv` into DuckDB
- Write 5 SQL files: revenue trends, cohort analysis, RFM segmentation, product & geo analysis

---
### Interview Questions You Can Now Answer:
1. *"How did you handle missing CustomerIDs?"* → Different decisions for different analyses
2. *"Did you delete the cancellation rows?"* → No — flagged them. Cancellation rate is a metric.
3. *"How much data did you remove and why?"* → X rows (Y%) — bad prices and duplicates only
4. *"What was your baseline revenue?"* → Check Step 9 output above